In [2]:
#!/usr/bin/env python3
r"""
Replicate a multi-animal DLC output tree as a SINGLE-animal tree (resident only).

For every file under SRC_ROOT it writes a counterpart under DEST_ROOT at the
same relative path:
  - .h5 / .csv : converted to single-animal format keeping ONLY the resident
                 individual (the 'individuals' level/row is dropped). Files that
                 are already single-animal are copied unchanged.
  - *_meta.pickle : copied as-is (small; carries the original video fps).
  - *_full.pickle : copied ONLY when no sibling *_meta.pickle exists (fps
                    fallback), since _full.pickle can be very large.
  - labeled videos and other files : skipped by default.

Originals under SRC_ROOT are never modified. Runs DRY_RUN first.
"""

import csv
import shutil
import sys
from pathlib import Path

# ============================================================================
# CONFIG
# ============================================================================

SRC_ROOT = r"C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions_topView_DLC_multiAnimal\trainingSessions_topView_DLC_multiAnimal"

DEST_ROOT = r"C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\unifiedDataset_topview_DLC_singleAnimal\analyzedVideos_trainingSessions_DLC_singleAnimal"

# Individual to keep (exact name from the DLC 'individuals' header).
RESIDENT_NAME = "Resident"

# Non-coordinate files:
COPY_META_PICKLE = True          # *_meta.pickle (has fps) -> always copy
COPY_FULL_PICKLE_IF_NO_META = True  # *_full.pickle -> copy only if no _meta sibling
COPY_LABELED_VIDEOS = False      # labeled .mp4/.avi -> skip
COPY_OTHER_FILES = False         # anything else (e.g. *_assemblies.pickle) -> skip

VIDEO_EXTS = (".mp4", ".avi", ".mov", ".mkv")

# Safety: preview first. Set to False to actually write.
DRY_RUN = False

# ============================================================================
# END CONFIG
# ============================================================================


def first_field_is_frame_index(field: str) -> bool:
    try:
        int(field)
        return True
    except (ValueError, TypeError):
        return False


def split_header(rows):
    n = 0
    for row in rows:
        if first_field_is_frame_index(row[0] if row else ""):
            break
        n += 1
    return rows[:n], rows[n:]


def individuals_row_index(header_rows):
    for i, row in enumerate(header_rows):
        if row and isinstance(row[0], str) and row[0].strip().lower() == "individuals":
            return i
    return None


def convert_csv(src: Path, dst: Path, resident: str):
    with src.open("r", newline="", encoding="utf-8") as f:
        rows = list(csv.reader(f))
    header, data = split_header(rows)
    ind_idx = individuals_row_index(header)
    if ind_idx is None:
        return {"action": "copy"}  # already single-animal
    ind_row = header[ind_idx]
    available = sorted({v.strip() for j, v in enumerate(ind_row) if j > 0 and v.strip()})
    if resident not in available:
        return {"action": "skip", "available": available}
    keep = [0] + [j for j, v in enumerate(ind_row) if j > 0 and v.strip() == resident]

    def sub(r):
        return [r[j] if j < len(r) else "" for j in keep]

    header = [sub(r) for k, r in enumerate(header) if k != ind_idx]
    data = [sub(r) for r in data]
    dst.parent.mkdir(parents=True, exist_ok=True)
    with dst.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerows(header)
        w.writerows(data)
    return {"action": "converted"}


def convert_h5(src: Path, dst: Path, resident: str):
    import pandas as pd
    df = pd.read_hdf(src)
    if "individuals" not in (df.columns.names or []):
        return {"action": "copy"}  # already single-animal
    inds = list(dict.fromkeys(df.columns.get_level_values("individuals")))
    if resident not in inds:
        return {"action": "skip", "available": inds}
    df = df.xs(resident, axis=1, level="individuals", drop_level=True)
    dst.parent.mkdir(parents=True, exist_ok=True)
    df.to_hdf(dst, key="df_with_missing", mode="w", format="table")
    return {"action": "converted"}


def classify(src: Path) -> str:
    """Return how to handle a file: convert_h5/convert_csv/copy/skip."""
    name = src.name.lower()
    ext = src.suffix.lower()
    if ext == ".h5":
        return "convert_h5"
    if ext == ".csv":
        return "convert_csv"
    if name.endswith("_meta.pickle"):
        return "copy" if COPY_META_PICKLE else "skip"
    if name.endswith("_full.pickle"):
        if COPY_FULL_PICKLE_IF_NO_META:
            base = src.name[: -len("_full.pickle")]
            if (src.parent / f"{base}_meta.pickle").exists():
                return "skip"          # meta covers fps; skip the big one
            return "copy"
        return "skip"
    if ext in VIDEO_EXTS:
        return "copy" if COPY_LABELED_VIDEOS else "skip"
    return "copy" if COPY_OTHER_FILES else "skip"


def main() -> None:
    src_root = Path(SRC_ROOT)
    dest_root = Path(DEST_ROOT)
    if not src_root.exists():
        print(f"[ERROR] SRC_ROOT does not exist: {src_root}")
        sys.exit(1)

    print(f"{'DRY RUN — nothing written' if DRY_RUN else 'WRITING'} "
          f"| resident='{RESIDENT_NAME}'\n")

    counts = {"converted": 0, "copy": 0, "skip": 0}
    missing_resident = []

    for src in sorted(src_root.rglob("*")):
        if src.is_dir():
            continue
        rel = src.relative_to(src_root)
        dst = dest_root / rel
        how = classify(src)

        if how == "skip":
            counts["skip"] += 1
            continue

        if how in ("convert_h5", "convert_csv"):
            if DRY_RUN:
                # Peek to report converted-vs-copy and catch bad resident names.
                res = (convert_csv if how == "convert_csv" else convert_h5)
                # dry check without writing: read only
                try:
                    if how == "convert_csv":
                        with src.open("r", newline="", encoding="utf-8") as f:
                            rows = list(csv.reader(f))
                        hdr, _ = split_header(rows)
                        ii = individuals_row_index(hdr)
                        if ii is None:
                            action = "copy"
                        else:
                            avail = sorted({v.strip() for j, v in enumerate(hdr[ii])
                                            if j > 0 and v.strip()})
                            action = "converted" if RESIDENT_NAME in avail else "skip"
                            if action == "skip":
                                missing_resident.append((str(rel), avail))
                    else:
                        import pandas as pd
                        df = pd.read_hdf(src)
                        if "individuals" not in (df.columns.names or []):
                            action = "copy"
                        else:
                            inds = list(dict.fromkeys(df.columns.get_level_values("individuals")))
                            action = "converted" if RESIDENT_NAME in inds else "skip"
                            if action == "skip":
                                missing_resident.append((str(rel), inds))
                except Exception as exc:  # noqa: BLE001
                    print(f"  [WARN] could not read {rel}: {exc}")
                    action = "skip"
            else:
                res = (convert_csv if how == "convert_csv" else convert_h5)(src, dst, RESIDENT_NAME)
                action = res["action"]
                if action == "skip":
                    missing_resident.append((str(rel), res.get("available", [])))
                elif action == "copy":
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(src, dst)

            if action == "skip":
                counts["skip"] += 1
            elif action == "copy":
                counts["copy"] += 1
                print(f"  [copy      ] {rel}  (already single-animal)")
            else:
                counts["converted"] += 1
                print(f"  [convert   ] {rel}")
            continue

        # plain copy (pickles / other kept files)
        if not DRY_RUN:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
        counts["copy"] += 1
        print(f"  [copy      ] {rel}")

    print(f"\n{'Planned' if DRY_RUN else 'Done'}: "
          f"{counts['converted']} converted, {counts['copy']} copied, "
          f"{counts['skip']} skipped.")
    if missing_resident:
        print(f"\n[WARN] '{RESIDENT_NAME}' not found in {len(missing_resident)} "
              f"file(s). First few, with the individuals actually present:")
        for rel, avail in missing_resident[:5]:
            print(f"    {rel}: {avail}")
        print("  -> fix RESIDENT_NAME to match, then re-run.")


if __name__ == "__main__":
    main()

WRITING | resident='Resident'

  [copy      ] mouse1010819\topView_DLCtracking_pcutoff_0.8_skeleton\mouse1010819_Day02_topView_compDLC_Resnet101_trainingAggression_trainingSessions_topViewFeb3shuffle2_snapshot_best-100_meta.pickle
  [convert   ] mouse1010819\topView_DLCtracking_pcutoff_0.8_skeleton\mouse1010819_Day02_topView_compDLC_Resnet101_trainingAggression_trainingSessions_topViewFeb3shuffle2_snapshot_best-100_sk.h5
  [convert   ] mouse1010819\topView_DLCtracking_pcutoff_0.8_skeleton\mouse1010819_Day02_topView_compDLC_Resnet101_trainingAggression_trainingSessions_topViewFeb3shuffle2_snapshot_best-100_sk_filtered.csv
  [convert   ] mouse1010819\topView_DLCtracking_pcutoff_0.8_skeleton\mouse1010819_Day02_topView_compDLC_Resnet101_trainingAggression_trainingSessions_topViewFeb3shuffle2_snapshot_best-100_sk_filtered.h5
  [copy      ] mouse1010819\topView_DLCtracking_pcutoff_0.8_skeleton\mouse1010819_Day04_topView_compDLC_Resnet101_trainingAggression_trainingSessions_topViewFeb3shuffle